# ML versus QML model comparison

**Team CtrlQ | Verified exploratory report | 22 October 2026**

This notebook is the complete report for the equal-sample ML versus QML experiment: five classical models, four statevector quantum-kernel SVM variants, and a majority-class dummy. All ten models use the same sampled observations in each partition.

The primary test period is 2023–2024. The 2025 onward period is a robustness check because it has already been inspected. IBM Quantum hardware is reserved for a later audit.

## Executive summary

On the **binary** primary test, the Pauli X/ZZ linear-entanglement QSVM reached **51.98% balanced accuracy**. The classical model chosen on validation, logistic regression, reached **45.71%**; the dummy baseline reached **50.00%**. The paired difference was **+6.27 percentage points**, with an exploratory 95% 10-date block-bootstrap interval of **+1.12 to +11.15 points**.

On the **three-class** primary test, the Z-map QSVM reached **38.55% balanced accuracy**. The validation-selected classical RBF SVM reached **37.03%**; the dummy reached **33.33%**. The paired difference was **+1.53 points**, with an exploratory interval of **+1.10 to +4.25 points**.

These are **exploratory findings**. The highlighted QML variants are the best test performers among four examined variants; the intervals were not adjusted for that selection or other comparisons. Stock observations on a date share market conditions. The 2025 onward period was previously inspected. This experiment does not establish quantum advantage or trading profitability.

## Experimental contract

- Twenty fixed stocks: the original five-stock universe expanded fourfold.
- Five classical models: logistic regression, linear SVM, RBF SVM, random forest, and histogram gradient boosting. Four quantum-kernel SVM feature-map variants: Z; ZZ linear; Pauli X/ZZ linear; Pauli Z/ZZ full.
- A majority-class dummy is a reference baseline.
- Features and labels are built causally from adjusted close data.
- Model hyperparameters are selected on 2021-2022 validation data only.
- 2023-2024 is the primary evaluation; 2025 onward is an exploratory robustness period.
- Balanced accuracy and macro-F1 are primary metrics because class proportions are not guaranteed to be equal.
- All models use the same 1,599 training, 800 validation, 1,200 primary-test, and 1,200 robustness observations. The development refit uses 2,399 observations.
- Sixteen engineered inputs are standardized, reduced to four PCA components, and angle-scaled. Each QML circuit has four qubits.
- QML kernels are exact statevector fidelities; no quantum hardware or finite-shot simulation was used.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RESULTS = ROOT / 'results' / 'complete_benchmark'
RESULTS.mkdir(parents=True, exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')
print('Project root:', ROOT)
print('Results directory:', RESULTS)

## Reproduce the benchmark

The benchmark is implemented in `experiments/complete_benchmark.py`. Set `RUN_BENCHMARK = True` to regenerate both task tables. Four large exact kernels may take hours. Saved results from older configurations are rejected below.

In [ ]:
import runpy
benchmark = runpy.run_path(str(ROOT / 'experiments' / 'complete_benchmark.py'))
RUN_BENCHMARK = False
if RUN_BENCHMARK:
    for task_name in ('binary', 'three_class'):
        benchmark['run'](ROOT, task_name)
        print('Completed:', task_name)
else:
    print('Using saved benchmark outputs. Set RUN_BENCHMARK=True to regenerate.')

In [ ]:
tables = {}
metadata = {}
for task_name in ('binary', 'three_class'):
    tables[task_name] = pd.read_csv(RESULTS / f'{task_name}_metrics.csv')
    metadata[task_name] = json.loads((RESULTS / f'{task_name}_metadata.json').read_text())
    expected = {'qsvm_z', 'qsvm_zz_linear', 'qsvm_pauli_linear', 'qsvm_pauli_full', 'matched_dummy', 'matched_logistic', 'matched_linear_svm', 'matched_rbf_svm', 'matched_random_forest', 'matched_hist_gradient_boosting'}
    assert len(metadata[task_name]['tickers']) == 20 and expected == set(tables[task_name].model), 'Saved results are stale; set RUN_BENCHMARK=True and rerun.'
    allowed_code_hashes = {hashlib.sha256(path.read_bytes()).hexdigest() for path in (ROOT / 'experiments' / 'complete_benchmark.py', ROOT / 'notebooks' / '09_equal_sample_benchmark.ipynb') if path.exists()}
    assert metadata[task_name]['code_sha256'] in allowed_code_hashes, 'Code changed since results were produced; rerun benchmark.'
    assert (RESULTS / f'{task_name}_predictions.csv').exists() and (RESULTS / f'{task_name}_verification.json').exists(), 'Verification files missing; rerun benchmark.'
    report = json.loads((RESULTS / f'{task_name}_verification.json').read_text())
    assert report['matched_test_rows'] == 1200 and report['verified_metric_values'] == 6 * len(tables[task_name])
    print(f'{task_name}:')
    display(tables[task_name].round(4))

## Data and split audit

The table below reports the exact rows and dates used by every model. Training labels do not cross the validation boundary, and validation labels do not cross the primary-test boundary. The 2025 onward split is labeled `confirmatory` in the saved file schema for compatibility; scientifically it is a robustness period.

In [ ]:
split_rows = []
for task_name, info in metadata.items():
    for split_name, split_info in info['splits'].items():
        split_rows.append({'task': task_name, 'split': split_name, **split_info})
split_table = pd.DataFrame(split_rows)
display(split_table)
assert all(split_table.rows > 0)
assert len(metadata['binary']['tickers']) == 20
assert metadata['binary']['features'] == metadata['three_class']['features']

## Primary test comparison

The dashed line is the constant-prediction balanced-accuracy reference: 0.50 for binary and 1/3 for three-class classification. Each model's hyperparameters were chosen on validation only. The highlighted best QML test model is descriptive; selecting it after viewing test performance adds selection bias.

In [ ]:
order = ['matched_dummy', 'matched_logistic', 'matched_linear_svm', 'matched_rbf_svm', 'matched_random_forest', 'matched_hist_gradient_boosting', 'qsvm_z', 'qsvm_zz_linear', 'qsvm_pauli_linear', 'qsvm_pauli_full']
fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
for ax, task_name in zip(axes, ('binary', 'three_class')):
    table = tables[task_name].set_index('model').reindex(order).reset_index()
    colors = ['#6c757d' if m == 'matched_dummy' else ('#2a9d8f' if m.startswith('qsvm_') else '#4472C4') for m in table.model]
    ax.bar(table['model'], table['test_balanced_accuracy'], color=colors)
    chance = 0.5 if task_name == 'binary' else 1/3
    ax.axhline(chance, color='black', linestyle='--', label=f'chance = {chance:.3f}')
    ax.set_title(task_name.replace('_', ' ').title())
    ax.set_xlabel('')
    ax.set_ylabel('Primary-test balanced accuracy')
    ax.tick_params(axis='x', rotation=35)
    ax.legend()
plt.show()

In [ ]:
summary_cols = ['model', 'validation_balanced_accuracy', 'test_balanced_accuracy', 'test_macro_f1', 'confirmatory_balanced_accuracy', 'confirmatory_macro_f1']
for task_name in ('binary', 'three_class'):
    print(task_name)
    display(tables[task_name].set_index('model').reindex(order)[summary_cols[1:]].round(4))

## Stability from primary test to later robustness period

A model that changes rank substantially between these periods is not robust evidence of superiority. The 2025 onward period has already been inspected, so these results remain exploratory.

In [ ]:
stability = []
for task_name, table in tables.items():
    for _, row in table.iterrows():
        stability.append({'task': task_name, 'model': row['model'],
                          'primary_test': row['test_balanced_accuracy'],
                          'confirmatory': row['confirmatory_balanced_accuracy'],
                          'change_pp': 100 * (row['confirmatory_balanced_accuracy'] - row['test_balanced_accuracy'])})
stability = pd.DataFrame(stability)
display(stability.pivot(index='model', columns='task', values='change_pp').reindex(order).round(2))
fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
for ax, task_name in zip(axes, ('binary', 'three_class')):
    part = stability[stability.task == task_name].set_index('model').reindex(order)
    ax.bar(part.index, part.change_pp, color='#70AD47')
    ax.axhline(0, color='black', linewidth=1)
    ax.set_title(task_name.replace('_', ' ').title())
    ax.set_ylabel('Later minus primary BA (percentage points)')
    ax.tick_params(axis='x', rotation=35)
plt.show()

## Paired uncertainty and resource notes

Every ML and QML model uses the same 20-stock observations on 80/40/60/60 sampled dates for training, validation, test and robustness. No full-sample classical models are included in this benchmark. Kernel construction grows quadratically with development size.

The QSVM is a classical SVC with a statevector fidelity kernel. Paired 95% intervals use 10-date moving blocks and retain all stocks on each sampled date. They are exploratory uncertainty summaries, not proof of superiority or quantum advantage.

In [ ]:
resource_table = pd.DataFrame({
    'item': ['ML development rows', 'QML development rows', 'PCA components / qubits', 'quantum execution'],
    'value': [metadata['binary']['splits']['train']['rows'] + metadata['binary']['splits']['validation']['rows'],
              int(tables['binary'].loc[tables['binary'].model == 'qsvm_zz_linear', 'qsvm_train_rows'].iloc[0]),
              4, 'exact noiseless statevector simulation']
})
display(resource_table)

In [ ]:
paired_rows = []
for task_name in ('binary', 'three_class'):
    report = json.loads((RESULTS / f'{task_name}_verification.json').read_text())
    for qml_model, interval in report['paired_date_block_comparisons'].items():
        paired_rows.append({'task': task_name, 'classical_comparator': report['validation_selected_classical'], 'qml_model': qml_model, **interval})
display(pd.DataFrame(paired_rows).round(2))

## Audit summary and selected comparisons

Each task passed 60 independent metric checks: three metrics × two evaluation periods × ten models. The verifier also compared stock-date keys and actual labels across all ten models. The classical comparator in the paired table was selected by validation balanced accuracy; no primary-test scores entered that selection.

In [ ]:
audit_rows = []
for task_name in ('binary', 'three_class'):
    report = json.loads((RESULTS / f'{task_name}_verification.json').read_text())
    audit_rows.append({'task': task_name, 'verified_metrics': report['verified_metric_values'],
                       'shared_test_rows': report['matched_test_rows'],
                       'shared_test_dates': report['matched_test_dates'],
                       'classical_comparator': report['validation_selected_classical']})
display(pd.DataFrame(audit_rows))
assert all(x['verified_metrics'] == 60 for x in audit_rows)

## Class balance and error patterns

Balanced accuracy averages recall across classes. The class counts and confusion matrices below show what that average hides. Rows of each confusion matrix are actual classes; columns are predicted classes. The matrices include the validation-selected classical comparator, the highest-scoring test QML variant, and the dummy.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
predictions = {task: pd.read_csv(RESULTS / f'{task}_predictions.csv', parse_dates=['Date']) for task in ('binary', 'three_class')}
for task_name in ('binary', 'three_class'):
    frame = predictions[task_name]
    report = json.loads((RESULTS / f'{task_name}_verification.json').read_text())
    classical = report['validation_selected_classical']
    qml = tables[task_name].loc[tables[task_name].model.str.startswith('qsvm_')].sort_values('test_balanced_accuracy', ascending=False).iloc[0]['model']
    print(f'{task_name}: selected classical = {classical}; highest test QML = {qml}')
    base = frame.loc[(frame.period == 'test') & (frame.model == classical)]
    display(base.actual.value_counts().sort_index().rename('test_rows').to_frame())
    labels = [0, 1] if task_name == 'binary' else [0, 1, 2]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
    for ax, model_name in zip(axes, ['matched_dummy', classical, qml]):
        part = frame.loc[(frame.period == 'test') & (frame.model == model_name)]
        cm = confusion_matrix(part.actual, part.prediction, labels=labels)
        ax.imshow(cm, cmap='Blues')
        for (i, j), value in np.ndenumerate(cm): ax.text(j, i, str(value), ha='center', va='center')
        ax.set_xticks(range(len(labels)), labels); ax.set_yticks(range(len(labels)), labels)
        ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(model_name)
    plt.show()
    part = frame.loc[(frame.period == 'test') & (frame.model == qml)]
    display(pd.DataFrame(classification_report(part.actual, part.prediction, labels=labels, output_dict=True, zero_division=0)).T.round(3))

## Computation time

The saved `fit_seconds` field is a single-machine measurement. For QML it includes statevector and kernel construction plus fitting; for classical models it is the final refit only, excluding their grid search. It is useful for describing this run, not a fair end-to-end speed comparison.

In [ ]:
for task_name in ('binary', 'three_class'):
    print(task_name)
    display(tables[task_name][['model', 'fit_seconds']].sort_values('fit_seconds').round(2))

## Interpretation and limitations

1. **Binary:** Pauli X/ZZ linear was the strongest QML configuration on the primary test at 51.98% balanced accuracy. The validation-selected logistic regression scored 45.71%; the dummy scored 50.00%. The gain over the selected comparator is larger than the gain over the dummy.
2. **Three-class:** Z-map QSVM was the strongest QML configuration on the primary test at 38.55%, versus 37.03% for the validation-selected RBF SVM and 33.33% for the dummy.
3. The exploratory paired 10-date block intervals for those two highlighted comparisons exclude zero. They do not account for examining four QML variants, two tasks, and many hyperparameters. The block width is one reasonable choice and should be varied in a sensitivity study.
4. All models saw identical sampled rows and the same four-component representation. This controls the row and input-dimension comparison. The 20-stock universe comes from a full-history eligibility filter and may suffer survivorship bias; source price-adjustment accuracy has not been independently verified.
5. The 2025 onward period was previously inspected and remains a robustness check. Neither period supports an untouched confirmatory claim. The QSVM kernels were computed on a classical four-qubit statevector simulator; the experiment does not establish quantum computational advantage or trading profitability.

### Recommended next experiment

Freeze one classical comparator, one QML feature map, feature definitions, sample policy, and primary metric before collecting a new later period. Report paired date-block intervals with several block widths and correct for the number of planned comparisons. Review historical stock membership and the price-adjustment policy. IBM hardware can later check a small fixed set of kernel circuits.